In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.signal import argrelextrema
from scipy import signal

# #### CONTROLS ####
version = 75
# # Get the directory where this script is located
script_dir = os.getcwd()

graphs_dir = os.path.join(script_dir, 'graphs')

# # Build the full path to the CSV files
# arduino_file_path = os.path.join(script_dir, f"arduino_output{version}.txt")
# keyence_file_path = os.path.join(script_dir, f"keyence_output{version}.csv")

# # Import Arduino TXT file with 4 columns: time, magnetic_raw, magnetic_filtered, posic_encoder
# df_arduino = pd.read_csv(
#     arduino_file_path, 
#     header=None, 
#     delimiter=', ', 
#     # names=['time_ms', 'magnetic_raw_mm', 'magnetic_filtered_mm', 'posic_encoder_mm'], 
#     names=['time_ms', 'magnetic_raw_mm', 'posic_encoder_mm', "magnetic_wrapped"], 
#     engine='python'
# )

# # Strip whitespace from all columns
# for col in df_arduino.columns:
#     df_arduino[col] = df_arduino[col].astype(str).str.strip()

# # Convert all columns to numeric, treating '-' as NaN
# for col in df_arduino.columns:
#     df_arduino[col] = pd.to_numeric(df_arduino[col], errors='coerce')

# # # Remove rows where any measurement column is NaN
# # df_arduino = df_arduino.dropna(subset=['magnetic_raw_mm', 'magnetic_filtered_mm', 'posic_encoder_mm'])

# print(f"Arduino dataframe length {len(df_arduino)}")

# # Import Keyence CSV file
# df_keyence = pd.read_csv(keyence_file_path, header=None)

# # Find the column with the most non-null values (since Keyence outputs multiple columns, only the long one has the position data)
# column_lengths = df_keyence.count()
# longest_column_index = column_lengths.idxmax()

# # Create a new DataFrame with only the longest column
# df_keyence = pd.DataFrame({
#     'keyence_mm': df_keyence[longest_column_index]
# })

In [ ]:
def load_version_data(version: int, script_dir: str = None) -> pd.DataFrame:
    """
    Load, merge, and process Arduino and Keyence data for a given version number.

    Args:
        version: The version number to load (e.g., 75)
        script_dir: Directory containing the data files. Defaults to current working directory.

    Returns:
        A merged and processed DataFrame with time, keyence, encoder, and magnetic columns.
    """
    if script_dir is None:
        script_dir = os.getcwd()

    arduino_file_path = os.path.join(script_dir, f"arduino_output{version}.txt")
    keyence_file_path = os.path.join(script_dir, f"keyence_output{version}.csv")

    # Import Arduino TXT file
    df_arduino = pd.read_csv(
        arduino_file_path,
        header=None,
        delimiter=', ',
        names=['time_ms', 'magnetic_raw_mm', 'posic_encoder_mm', 'magnetic_wrapped'],
        engine='python'
    )

    # Strip whitespace and convert to numeric
    for col in df_arduino.columns:
        df_arduino[col] = pd.to_numeric(
            df_arduino[col].astype(str).str.strip(),
            errors='coerce'
        )

    # print(f"[v{version}] Arduino dataframe length: {len(df_arduino)}")

    # Import Keyence CSV file — keep only the column with the most non-null values
    df_keyence_raw = pd.read_csv(keyence_file_path, header=None)
    longest_column_index = df_keyence_raw.count().idxmax()
    df_keyence = pd.DataFrame({'keyence_mm': df_keyence_raw[longest_column_index]})

    # print(f"[v{version}] Keyence dataframe length: {len(df_keyence)}")

    # Merge (Keyence will usually have fewer rows)
    df_merged = pd.concat([df_arduino, df_keyence], axis=1)

    # Time column in seconds, zeroed from start
    df_merged['time_s'] = (df_merged['time_ms'] - df_merged['time_ms'].iloc[0]) / 1000.0

    # Zero and scale displacement values
    df_merged['keyence_mm'] = df_merged['keyence_mm'] * -1
    # df_merged['posic_encoder_mm'] = df_merged['posic_encoder_mm'] - df_merged['posic_encoder_mm'].iloc[0]
    # df_merged['magnetic_raw_mm'] = df_merged['magnetic_raw_mm'] - df_merged['magnetic_raw_mm'].iloc[0]
    df_merged['posic_encoder_mm'] = df_merged['posic_encoder_mm'] / 1000000 * 19.53125 * (-1) * 0.938
    df_merged['magnetic_raw'] = df_merged['magnetic_raw_mm']
    # df_merged['magnetic_raw_mm'] = df_merged['magnetic_raw_mm'] - df_merged['magnetic_raw_mm'].iloc[0]
    df_merged['magnetic_raw_mm'] = df_merged['magnetic_raw_mm'] * -1 * 0.00048828125 * 0.9459754389

    # print(f"[v{version}] Merged dataframe length: {len(df_merged)}")

    return df_merged

In [ ]:
def plot_sensor_data(
    version,
    *plot_configs,
    n_graphs=1,
    title="Sensor Data vs Time",
    xlabel="Time (s)",
    ylabel="Displacement (mm)",
    figsize_per_graph=(16, 6),
    output_filename=None
):
    """
    Flexible sensor data plotting function.

    Parameters:
    -----------
    version : int
        Version number for the plot title and filename.
    *plot_configs : dict
        Each dict defines one line to plot. Keys:
            - df         : pd.DataFrame (required)
            - y_col      : str, column to plot on y-axis (required)
            - x_col      : str, column to plot on x-axis (default: 'time_s')
            - label      : str, legend label (default: y_col)
            - color      : str (default: auto)
            - linewidth  : float (default: 1.5)
            - alpha      : float (default: 0.8)
            - graph_idx  : int, which subplot to plot on, 0-indexed (default: 0)
            - plot_type  : 'line' or 'scatter' (default: 'line')
    n_graphs : int
        Number of subplots (stacked vertically). Default: 1.
    title : str or list of str
        Title(s) for the plot(s). If a single string, used for the figure
        suptitle. If a list, each entry is the title for the corresponding subplot.
    xlabel : str or list of str
        X-axis label(s). Scalar applies to all subplots; list applies per subplot.
    ylabel : str or list of str
        Y-axis label(s). Scalar applies to all subplots; list applies per subplot.
    figsize_per_graph : tuple
        (width, height) per subplot. Total figure height scales with n_graphs.
    output_filename : str, optional
        Output filename (without directory). Defaults to
        f'sensor_data_v{version}.png'.

    Example:
    --------
    plot_sensor_data(
        version=1,
        {"df": df_merged, "y_col": "keyence_mm",      "label": "Keyence",       "color": "blue",  "graph_idx": 0},
        {"df": df_merged, "y_col": "magnetic_raw_mm",  "label": "Magnetic",      "color": "red",   "graph_idx": 0},
        {"df": df_merged, "y_col": "posic_encoder_mm", "label": "Posic Encoder", "color": "green", "graph_idx": 1},
        n_graphs=2,
        title=["Raw Signals", "Encoder"],
        ylabel=["Displacement (mm)", "Displacement (mm)"],
    )
    """

    def _to_list(val, n):
        """Broadcast scalar or list to length-n list."""
        if isinstance(val, list):
            return val
        return [val] * n

    titles = _to_list(title, n_graphs)
    xlabels = _to_list(xlabel, n_graphs)
    ylabels = _to_list(ylabel, n_graphs)

    fig_height = figsize_per_graph[1] * n_graphs
    fig, axes = plt.subplots(n_graphs, 1, figsize=(figsize_per_graph[0], fig_height))

    # Always work with a list of axes
    if n_graphs == 1:
        axes = [axes]

    # Apply per-subplot labels and titles
    for i, ax in enumerate(axes):
        ax.set_xlabel(xlabels[i], fontsize=12)
        ax.set_ylabel(ylabels[i], fontsize=12)
        ax.set_title(titles[i], fontsize=14)
        ax.grid(True, alpha=0.3)

    # Plot each config
    for cfg in plot_configs:
        df      = cfg["df"]
        y_col   = cfg["y_col"]
        x_col   = cfg.get("x_col", "time_s")
        label   = cfg.get("label", y_col)
        color   = cfg.get("color", None)
        lw      = cfg.get("linewidth", 1.5)
        alpha   = cfg.get("alpha", 0.8)
        idx     = cfg.get("graph_idx", 0)
        ptype   = cfg.get("plot_type", "line")

        ax = axes[idx]
        valid = df[y_col].notna()

        plot_kwargs = dict(color=color, label=label, alpha=alpha)

        if ptype == "scatter":
            ax.scatter(df.loc[valid, x_col], df.loc[valid, y_col], **plot_kwargs)
        else:
            ax.plot(df.loc[valid, x_col], df.loc[valid, y_col],
                    linewidth=lw, **plot_kwargs)

        ax.legend(fontsize=10, loc="best")

    # Overall suptitle only when n_graphs > 1 or title is a single string
    if n_graphs > 1:
        fig.suptitle(f"Version {version}", fontsize=13, y=1.01)

    plt.tight_layout()

    fname = output_filename or f"sensor_data_v{version}.png"
    output_path = os.path.join(graphs_dir, fname)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"\nPlot saved to: {output_path}")

In [ ]:
def compute_errors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute magnetic and positional encoder errors relative to Keyence reference.

    Args:
        df: A merged DataFrame returned by load_version_data()

    Returns:
        The same DataFrame with added 'magnetic_error' and 'posic_error' columns.
    """
    df["magnetic_error"] = df["keyence_mm"] - df["magnetic_raw_mm"]
    df["posic_error"] = df["keyence_mm"] - df["posic_encoder_mm"]
    return df

In [ ]:
def sync_dataframes(*dfs: pd.DataFrame, col: str = "magnetic_raw", search_rows: int = 20) -> list[pd.DataFrame]:
    """
    Sync multiple DataFrames to start at the same shared value in a given column.
    Compares the first `search_rows` rows of each DataFrame to find the first
    value that appears in all of them, then trims each DataFrame to start there.

    Args:
        *dfs: Two or more DataFrames to sync.
        col: Column to compare (default: 'magnetic_raw').
        search_rows: How many rows from the top to search (default: 20).

    Returns:
        A list of trimmed DataFrames, in the same order as input.
    """
    # Get the candidate values from the top N rows of each dataframe
    candidate_sets = [set(df[col].iloc[:search_rows].dropna()) for df in dfs]

    # Find values common to all dataframes
    shared_values = candidate_sets[0].intersection(*candidate_sets[1:])

    if not shared_values:
        raise ValueError(f"No shared values found in the first {search_rows} rows of '{col}' across all DataFrames.")

    # For each dataframe, find the first row index (within search window) that holds a shared value
    first_shared_indices = []
    for df in dfs:
        for i, val in enumerate(df[col].iloc[:search_rows]):
            if val in shared_values:
                first_shared_indices.append(i)
                break

    # The sync point is the shared value that appears latest (i.e. requires the most trimming)
    sync_pos = max(first_shared_indices)
    sync_value = dfs[first_shared_indices.index(sync_pos)][col].iloc[sync_pos]

    print(f"Syncing on value: {sync_value} (column: '{col}')")

    # Trim each dataframe to start at the sync value
    trimmed = []
    for df in dfs:
        start_idx = df[col].iloc[:search_rows][df[col].iloc[:search_rows] == sync_value].index[0]
        trimmed.append(df.loc[start_idx:].reset_index(drop=True))
        print(f"  Trimmed {start_idx} rows → new length: {len(trimmed[-1])}")

    return trimmed

In [ ]:
def normalize_to(df: pd.DataFrame, reference_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Normalize specified columns in df using the first row values of a reference DataFrame.

    Args:
        df: DataFrame to normalize.
        reference_df: DataFrame whose first-row values are used as the zero reference.
        columns: List of column names to normalize.

    Returns:
        The df with specified columns zeroed relative to reference_df's first row.
    """
    for col in columns:
        df[col] = df[col] - reference_df[col].iloc[0]
    return df

In [ ]:
def add_cycle_groups(*dfs: pd.DataFrame, time_col: str = "time_s", threshold: float = 0.5) -> list[pd.DataFrame]:
    """
    Adds a 'cycle_group' column to each DataFrame that increments only when
    the time difference between consecutive rows exceeds the threshold.

    Args:
        *dfs: One or more DataFrames to process.
        time_col: Column containing time in seconds (default: 'time_s').
        threshold: Time gap in seconds that triggers a new group (default: 0.5).

    Returns:
        A list of DataFrames with the new 'cycle_group' column added.
    """
    result = []
    for df in dfs:
        df = df.copy()
        time_diff = df[time_col].diff().abs()
        df["cycle_group"] = (time_diff > threshold).cumsum()
        result.append(df)
    return result

In [ ]:
# # Merge both keyence and arduino dataframe (keyence will usually have less data points)
# df_merged = pd.concat([df_arduino, df_keyence], axis=1)

# # Create time_s column by calculating cumulative time differences in seconds
# df_merged['time_s'] = (df_merged['time_ms'] - df_merged['time_ms'].iloc[0]) / 1000.0
# # Zero the displacement values
# df_merged['keyence_mm'] = (df_merged['keyence_mm'] - df_merged['keyence_mm'].iloc[0]) * -1
# df_merged['posic_encoder_mm'] = df_merged['posic_encoder_mm'] - df_merged['posic_encoder_mm'].iloc[0]
# df_merged['posic_encoder_mm'] = df_merged['posic_encoder_mm'] / 1000000 * 19.53125 * (-1) * 0.938
# df_merged['magnetic_raw'] = df_merged['magnetic_raw_mm']
# df_merged['magnetic_raw_mm'] = df_merged['magnetic_raw_mm'] - df_merged['magnetic_raw_mm'].iloc[0]
# df_merged['magnetic_raw_mm'] = df_merged['magnetic_raw_mm'] * -1 * 0.00048828125 * 0.9459754389
# # df_merged['magnetic_raw_mm'] = df_merged['magnetic_raw_mm'] * -1 * 0.00048828125

df_75 = load_version_data(75, script_dir)
df_76 = load_version_data(76, script_dir)
df_77 = load_version_data(77, script_dir)

df_75, df_76, df_77 = sync_dataframes(df_75, df_76, df_77, col="magnetic_raw", search_rows=20)

cols = ["keyence_mm", "posic_encoder_mm", "magnetic_raw_mm"]

df_75 = normalize_to(df_75, df_75, cols)
df_76 = normalize_to(df_76, df_76, cols)
df_77 = normalize_to(df_77, df_77, cols)

# Plot raw sensor data against time
cfg1 = {"df": df_75, "y_col": "keyence_mm",      "label": "Keyence",       "color": "blue"}
cfg2 = {"df": df_75, "y_col": "magnetic_raw_mm",  "label": "Magnetic",      "color": "red"}
cfg3 = {"df": df_75, "y_col": "posic_encoder_mm", "label": "Posic Encoder", "color": "green"}

plot_sensor_data(
    1,
    cfg1, cfg2, cfg3,
    n_graphs=1,
    title=f"Raw Sensor Data vs Time (Version {version})",
    output_filename=f"raw_sensor_data_v{version}.png"
)

plt.show()

In [ ]:
## ERROR PlOTS
df_75 = compute_errors(df_75)
df_76 = compute_errors(df_76)
df_77 = compute_errors(df_77)

cfg1 = {"df": df_75, "x_col": "magnetic_raw_mm", "y_col": "posic_error", "label": "Posic Error (V75)", "color": "blue"}
cfg2 = {"df": df_76, "x_col": "magnetic_raw_mm", "y_col": "posic_error", "label": "Posic Error (V76)",  "color": "red"}
cfg3 = {"df": df_77, "x_col": "magnetic_raw_mm", "y_col": "posic_error", "label": "Posic Error (V77)",  "color": "green"}

# cfg1 = {"df": df_75.iloc[:len(df_75)//2], "x_col": "keyence_mm", "y_col": "magnetic_error", "label": "Posic Error (V75)", "color": "blue"}
# cfg2 = {"df": df_76.iloc[:len(df_76)//2], "x_col": "keyence_mm", "y_col": "magnetic_error", "label": "Posic Error (V76)",  "color": "red"}
# cfg3 = {"df": df_77.iloc[:len(df_77)//2], "x_col": "keyence_mm", "y_col": "magnetic_error", "label": "Posic Error (V77)",  "color": "green"}

plot_sensor_data(
    1,
    cfg1, cfg2, cfg3,
    n_graphs=1,
    title=f"Error vs Ground Truth Displacement (Posic Encoder)",
    # title=f"Error vs Ground Truth Displacement (Version {version})",
    output_filename=f"error_data_v{version}.png",
    xlabel="Magnetic Raw Displacement (mm)"
)


In [ ]:
df_75, df_76, df_77 = add_cycle_groups(df_75, df_76, df_77)

## Note for this case, cycle_group = 0 is from 0 to 0.1 mm, 1 is 0.1 to 5 mm, and 2 is 5 to 0.1 mm
# Plot raw sensor data against time
cfg1 = {"df": df_75[df_75["cycle_group"] == 2], "x_col": "magnetic_raw_mm", "y_col": "magnetic_error", "label": "Magnetic Error (V75)",       "color": "blue"}
cfg2 = {"df": df_76[df_76["cycle_group"] == 2], "x_col": "magnetic_raw_mm", "y_col": "magnetic_error", "label": "Magnetic Error (V76)",      "color": "red"}
cfg3 = {"df": df_77[df_77["cycle_group"] == 2], "x_col": "magnetic_raw_mm", "y_col": "magnetic_error", "label": "Magnetic Error (V77)", "color": "green"}

plot_sensor_data(
    1,
    cfg1, cfg2, cfg3,
    n_graphs=1,
    title=f"Error vs Ground Truth Displacement Backward Motion (Magnetic Encoder)",
    output_filename=f"error_backward_magnetic.png"
)

In [ ]:
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt

# 2. Apply Savitzky-Golay Filter
#    - window_length: number of points used to fit each polynomial (must be odd)
#    - polyorder: degree of the polynomial (must be < window_length)
window_length = 11   # adjust: larger = smoother, must be odd
polyorder = 3        # adjust: typically 2–4

df_75_new = df_75[df_75["cycle_group"] == 2]

df_75_new["magnetic_error_smoothed"] = savgol_filter(
    df["magnetic_error"],
    window_length=window_length,
    polyorder=polyorder
)

# 3. Plot original vs smoothed
plt.figure(figsize=(12, 5))
plt.plot(df_75_new["magnetic_raw_mm"], df_75_new["magnetic_error"], alpha=0.5, label="Original", color="steelblue")
plt.plot(df_75_new["magnetic_raw_mm"], df_75_new["magnetic_error_smoothed"], label=f"Savitzky-Golay (w={window_length}, p={polyorder})", color="crimson", linewidth=2)
plt.xlabel("magnetic_raw_mm")
plt.ylabel("magnetic_error")
plt.title("Savitzky-Golay Filter")
plt.legend()
plt.tight_layout()
plt.show()

print("Finished running")